In [27]:
import warnings
import pandas as pd
import numpy as np 
from pathlib import Path
from scipy.stats import linregress

In [12]:


warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style"
)

ruta_ine = Path("../Data")

# 1. Población por edad y sexo
df_poblacion = pd.read_excel(
    ruta_ine / "2026-07-20_INE_2026_2036_36643.raw.xlsx",
    header=6
)

# 2. Saldo vegetativo
df_saldo_vegetativo = pd.read_excel(
    ruta_ine / "2026-07-20_INE_saldo_vegetativo_2026_2075.xlsx",
    header=None,
    skiprows=6,
    names=[
        "periodo",
        "saldo_vegetativo_por_mil"
    ]
)

# Convertir ambas columnas a números
df_saldo_vegetativo["periodo"] = pd.to_numeric(
    df_saldo_vegetativo["periodo"],
    errors="coerce"
)

df_saldo_vegetativo["saldo_vegetativo_por_mil"] = pd.to_numeric(
    df_saldo_vegetativo["saldo_vegetativo_por_mil"],
    errors="coerce"
)

# Seleccionar solo 2026-2036
df_saldo_vegetativo = (
    df_saldo_vegetativo.loc[
        df_saldo_vegetativo["periodo"].between(
            2026,
            2036
        )
    ]
    .copy()
)

# Año como número entero
df_saldo_vegetativo["periodo"] = (
    df_saldo_vegetativo["periodo"].astype(int)
)

# Ordenar cronológicamente
df_saldo_vegetativo = (
    df_saldo_vegetativo
    .sort_values("periodo")
    .reset_index(drop=True)
)


# 3. Migración neta
df_migracion_neta = pd.read_excel(
    ruta_ine / "2026-07-20_INE_migracion_neta_2026_2075.xlsx",
    header=None,
    skiprows=6,
    names=[
        "periodo",
        "tasa_migracion_neta"
    ]
)

# convertir datos
df_migracion_neta["periodo"] = pd.to_numeric(
    df_migracion_neta["periodo"],
    errors="coerce"
)

df_migracion_neta["tasa_migracion_neta"] = pd.to_numeric(
    df_migracion_neta["tasa_migracion_neta"],
    errors="coerce"
)

# decada ordenada
df_migracion_neta = (
    df_migracion_neta.loc[
        df_migracion_neta["periodo"].between(
            2026,
            2036
        )
    ]
    .copy()
)

df_migracion_neta["periodo"] = (
    df_migracion_neta["periodo"].astype(int)
)

df_migracion_neta = (
    df_migracion_neta
    .sort_values("periodo")
    .reset_index(drop=True)
)


display(df_poblacion.head())
display(df_saldo_vegetativo.head())
display(df_migracion_neta.head())

,,2076,2075,2074,2073,2072,2071,2070,2069,2068,...,2035,2034,2033,2032,2031,2030,2029,2028,2027,2026
0,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Todas las edades,5.297299e+07,5.302801e+07,5.308861e+07,5.315315e+07,5.322143e+07,5.329357e+07,5.336974e+07,5.344987e+07,5.353358e+07,...,5.288414e+07,5.265876e+07,5.240291e+07,5.211205e+07,5.178245e+07,5.141203e+07,5.100118e+07,5.055393e+07,5.007877e+07,49590099.0
2,0 años,3.555469e+05,3.548414e+05,3.539143e+05,3.527670e+05,3.514238e+05,3.499194e+05,3.482985e+05,3.466060e+05,3.448875e+05,...,3.613875e+05,3.569122e+05,3.524054e+05,3.478817e+05,3.433373e+05,3.387731e+05,3.342223e+05,3.297576e+05,3.255249e+05,323425.0
3,1 año,3.584631e+05,3.575461e+05,3.564309e+05,3.551247e+05,3.536552e+05,3.520659e+05,3.504023e+05,3.487126e+05,3.470484e+05,...,3.621268e+05,3.579681e+05,3.538438e+05,3.497394e+05,3.456374e+05,3.415435e+05,3.374931e+05,3.335851e+05,3.316380e+05,324651.0
4,2 años,3.611875e+05,3.600844e+05,3.588126e+05,3.573823e+05,3.558296e+05,3.541985e+05,3.525382e+05,3.509025e+05,3.493458e+05,...,3.632257e+05,3.594418e+05,3.557263e+05,3.520507e+05,3.484016e+05,3.447869e+05,3.412688e+05,3.395938e+05,3.329214e+05,330814.0


,periodo,saldo_vegetativo_por_mil
0,2026,-2.542558
1,2027,-2.529663
2,2028,-2.517906
3,2029,-2.507789
4,2030,-2.504546


,periodo,tasa_migracion_neta
0,2026,12.348410
1,2027,11.973166
2,2028,11.325929
3,2029,10.531064
4,2030,9.683639


In [13]:
print("Población:", df_poblacion.shape)
print("Saldo vegetativo:", df_saldo_vegetativo.shape)
print("Migración neta:", df_migracion_neta.shape)

Población: (315, 52)
Saldo vegetativo: (11, 2)
Migración neta: (11, 2)


In [14]:
# renombra columna edad
df_poblacion = df_poblacion.rename(columns={df_poblacion.columns[0]: "edad"})

# limpiar espacios
df_poblacion["edad"] = df_poblacion["edad"].astype(str).str.strip()

# bloque total
inicio = df_poblacion.index[df_poblacion["edad"] == "Todas las edades"][0]
fin = df_poblacion.index[df_poblacion["edad"] == "Hombres"][0]

df_total = df_poblacion.loc[inicio:fin - 1].copy()

# decada
df_poblacion.columns = df_poblacion.columns.astype(str)

columnas = ["edad"] + [str(año) for año in range(2026, 2037)]

df_total = df_total[columnas]

display(df_total.head())


,edad,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036
1,Todas las edades,49590099.0,5.007877e+07,5.055393e+07,5.100118e+07,5.141203e+07,5.178245e+07,5.211205e+07,5.240291e+07,5.265876e+07,5.288414e+07,5.308394e+07
2,0 años,323425.0,3.255249e+05,3.297576e+05,3.342223e+05,3.387731e+05,3.433373e+05,3.478817e+05,3.524054e+05,3.569122e+05,3.613875e+05,3.657939e+05
3,1 año,324651.0,3.316380e+05,3.335851e+05,3.374931e+05,3.415435e+05,3.456374e+05,3.497394e+05,3.538438e+05,3.579681e+05,3.621268e+05,3.663093e+05
4,2 años,330814.0,3.329214e+05,3.395938e+05,3.412688e+05,3.447869e+05,3.484016e+05,3.520507e+05,3.557263e+05,3.594418e+05,3.632257e+05,3.670972e+05
5,3 años,350186.0,3.387151e+05,3.406227e+05,3.468626e+05,3.482119e+05,3.513316e+05,3.545363e+05,3.577899e+05,3.611029e+05,3.644998e+05,3.680141e+05


In [15]:
df_poblacion_edades = df_total.copy()

# Extraer edad numérica
df_poblacion_edades["edad_num"] = (
    df_poblacion_edades["edad"]
    .str.extract(r"(\d+)")
    .astype(float)
)

# Seleccionar población de 18 a 79 años
df_poblacion_edades = (
    df_poblacion_edades.loc[
        df_poblacion_edades["edad_num"]
        .between(18, 79)
    ]
    .copy()
)

df_poblacion_edades["edad_num"] = (
    df_poblacion_edades["edad_num"].astype(int)
)

# Años que queremos analizar
años = [str(año) for año in range(2026, 2037)]

# Crear una nueva tabla en formato largo
df_poblacion_larga = df_poblacion_edades.melt(
    id_vars=["edad_num"],
    value_vars=años,
    var_name="periodo",
    value_name="poblacion"
)

# Convertir tipos
df_poblacion_larga["periodo"] = (
    df_poblacion_larga["periodo"].astype(int)
)

df_poblacion_larga["poblacion"] = pd.to_numeric(
    df_poblacion_larga["poblacion"],
    errors="coerce"
)

# Ordenar
df_poblacion_larga = (
    df_poblacion_larga
    .sort_values(["periodo", "edad_num"])
    .reset_index(drop=True)
)

display(df_poblacion_larga.head())

,edad_num,periodo,poblacion
0,18,2026,551342.0
1,19,2026,559068.0
2,20,2026,553819.0
3,21,2026,557615.0
4,22,2026,557311.0


In [17]:
print("Dimensiones:", df_poblacion_larga.shape)
print("Edad mínima:", df_poblacion_larga["edad_num"].min())
print("Edad máxima:", df_poblacion_larga["edad_num"].max())
print("Periodo mínimo:", df_poblacion_larga["periodo"].min())
print("Periodo máximo:", df_poblacion_larga["periodo"].max())

Dimensiones: (682, 3)
Edad mínima: 18
Edad máxima: 79
Periodo mínimo: 2026
Periodo máximo: 2036


In [18]:
print(df_poblacion.shape)
print(df_saldo_vegetativo.shape)
print(df_migracion_neta.shape)

(315, 52)
(11, 2)
(11, 2)


## Segmentación por grupos de edad

Las proyecciones del INE proporcionan la población desagregada por cada edad individual (18, 19, 20, ..., 79 años). Sin embargo, para responder a la pregunta de negocio resulta más útil agrupar las edades en segmentos que representen diferentes etapas del ciclo de vida.

Esta segmentación permite analizar cómo evolucionará el tamaño de cada grupo de clientes potenciales durante los próximos años.

Los segmentos definidos son:

- **18-24 años:** Jóvenes.
- **25-34 años:** Jóvenes adultos.
- **35-44 años:** Adultos.
- **45-54 años:** Adultos consolidados.
- **55-64 años:** Prejubilados.
- **65-79 años:** Sénior.

Estos grupos reflejan diferentes necesidades financieras y permiten relacionar posteriormente la información demográfica con el comportamiento financiero obtenido de la Encuesta de Competencias Financieras (ECF).

Una vez creada la variable *segmento_edad*, se agrupa la población por año y segmento para conocer cuántas personas pertenecen a cada grupo en cada periodo.

In [19]:
# Copia del dataframe
df_segmentos = df_poblacion_larga.copy()

# Crear segmentos de edad
df_segmentos["segmento_edad"] = pd.cut(
    df_segmentos["edad_num"],
    bins=[17,24,34,44,54,64,79],
    labels=[
        "18-24 Jóvenes",
        "25-34 Jóvenes adultos",
        "35-44 Adultos",
        "45-54 Adultos consolidados",
        "55-64 Prejubilados",
        "65-79 Sénior"
    ]
)

# Agrupar población por año y segmento
df_segmentos_edad = (
    df_segmentos
    .groupby(["periodo","segmento_edad"], observed=True)["poblacion"]
    .sum()
    .reset_index()
)

display(df_segmentos_edad.head())

,periodo,segmento_edad,poblacion
0,2026,18-24 Jóvenes,3878454.0
1,2026,25-34 Jóvenes adultos,5689845.0
2,2026,35-44 Adultos,6526108.0
3,2026,45-54 Adultos consolidados,8071195.0
4,2026,55-64 Prejubilados,7122580.0


## Peso relativo de cada segmento poblacional

Conocer el número absoluto de personas en cada segmento resulta útil, pero no permite apreciar fácilmente cómo cambia la estructura de la población.

Por ello se calcula el peso porcentual que representa cada segmento respecto al total de la población proyectada para cada año.

Este indicador permite responder preguntas como:

- ¿Qué porcentaje de la población serán jóvenes dentro de diez años?
- ¿Aumentará el peso de la población sénior?
- ¿Qué segmentos perderán importancia relativa?

Este análisis facilita comparar años diferentes aunque la población total aumente o disminuya.

In [20]:
totales = (
    df_segmentos_edad
    .groupby("periodo")["poblacion"]
    .sum()
    .rename("total")
)

df_segmentos_edad = (
    df_segmentos_edad
    .merge(totales, on="periodo")
)

df_segmentos_edad["porcentaje"] = (
    df_segmentos_edad["poblacion"]
    / df_segmentos_edad["total"]
    *100
)

display(df_segmentos_edad)

,periodo,segmento_edad,poblacion,total,porcentaje
0,2026,18-24 Jóvenes,3.878454e+06,3.860349e+07,10.046901
1,2026,25-34 Jóvenes adultos,5.689845e+06,3.860349e+07,14.739200
2,2026,35-44 Adultos,6.526108e+06,3.860349e+07,16.905488
3,2026,45-54 Adultos consolidados,8.071195e+06,3.860349e+07,20.907943
4,2026,55-64 Prejubilados,7.122580e+06,3.860349e+07,18.450613
...,...,...,...,...,...
61,2036,25-34 Jóvenes adultos,6.850573e+06,4.197480e+07,16.320680
62,2036,35-44 Adultos,6.689109e+06,4.197480e+07,15.936013
63,2036,45-54 Adultos consolidados,7.101191e+06,4.197480e+07,16.917748
64,2036,55-64 Prejubilados,8.098875e+06,4.197480e+07,19.294614


# Crecimiento absoluto y crecimiento porcentual
## Evolución de los segmentos poblacionales

Una vez definidos los segmentos, se analiza cómo evolucionan entre el primer y el último año disponible en la proyección.

Se calculan dos indicadores:

- **Crecimiento absoluto:** diferencia en el número de personas entre ambos años.
- **Crecimiento porcentual:** porcentaje de crecimiento o disminución respecto al año inicial.

El crecimiento absoluto muestra cuántas personas gana o pierde cada segmento.

El crecimiento porcentual permite comparar segmentos de distinto tamaño, identificando cuáles experimentan un mayor crecimiento relativo.

Estos indicadores permiten detectar qué grupos de población aumentarán y cuáles disminuirán, información fundamental para decidir si es necesario redefinir los segmentos de clientes de la empresa.

# Crecimiento absoluto

In [21]:
df_crecimiento = (
    df_segmentos_edad
    .pivot(
        index="segmento_edad",
        columns="periodo",
        values="poblacion"
    )
)

df_crecimiento["crecimiento_absoluto"] = (
    df_crecimiento[2036]
    - df_crecimiento[2026]
)

display(df_crecimiento)

periodo,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,crecimiento_absoluto
segmento_edad,,,,,,,,,,,,
18-24 Jóvenes,3878454.0,4.002649e+06,4.100102e+06,4.166547e+06,4.206732e+06,4.223637e+06,4.195619e+06,4.165050e+06,4.105946e+06,4.051151e+06,3.980563e+06,1.021087e+05
25-34 Jóvenes adultos,5689845.0,5.804328e+06,5.913587e+06,6.044268e+06,6.184294e+06,6.321412e+06,6.458615e+06,6.577189e+06,6.706231e+06,6.792953e+06,6.850573e+06,1.160728e+06
35-44 Adultos,6526108.0,6.479453e+06,6.466621e+06,6.473977e+06,6.482370e+06,6.500020e+06,6.525927e+06,6.562390e+06,6.590409e+06,6.632945e+06,6.689109e+06,1.630013e+05
45-54 Adultos consolidados,8071195.0,8.075960e+06,8.045868e+06,7.980870e+06,7.880490e+06,7.755446e+06,7.608178e+06,7.463157e+06,7.322954e+06,7.206143e+06,7.101191e+06,-9.700041e+05
55-64 Prejubilados,7122580.0,7.256752e+06,7.389719e+06,7.505974e+06,7.609673e+06,7.728139e+06,7.848202e+06,7.938794e+06,8.025321e+06,8.074426e+06,8.098875e+06,9.762954e+05
65-79 Sénior,7315305.0,7.496579e+06,7.675785e+06,7.840684e+06,8.054623e+06,8.271145e+06,8.481158e+06,8.681559e+06,8.873134e+06,9.070451e+06,9.254488e+06,1.939183e+06


# Crecimiento porcentual

In [22]:
df_crecimiento["crecimiento_%"] = (
    (
        df_crecimiento[2036]
        - df_crecimiento[2026]
    )
    /
    df_crecimiento[2026]
    *100
)

display(df_crecimiento)

periodo,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,crecimiento_absoluto,crecimiento_%
segmento_edad,,,,,,,,,,,,,
18-24 Jóvenes,3878454.0,4.002649e+06,4.100102e+06,4.166547e+06,4.206732e+06,4.223637e+06,4.195619e+06,4.165050e+06,4.105946e+06,4.051151e+06,3.980563e+06,1.021087e+05,2.632716
25-34 Jóvenes adultos,5689845.0,5.804328e+06,5.913587e+06,6.044268e+06,6.184294e+06,6.321412e+06,6.458615e+06,6.577189e+06,6.706231e+06,6.792953e+06,6.850573e+06,1.160728e+06,20.399985
35-44 Adultos,6526108.0,6.479453e+06,6.466621e+06,6.473977e+06,6.482370e+06,6.500020e+06,6.525927e+06,6.562390e+06,6.590409e+06,6.632945e+06,6.689109e+06,1.630013e+05,2.497680
45-54 Adultos consolidados,8071195.0,8.075960e+06,8.045868e+06,7.980870e+06,7.880490e+06,7.755446e+06,7.608178e+06,7.463157e+06,7.322954e+06,7.206143e+06,7.101191e+06,-9.700041e+05,-12.018098
55-64 Prejubilados,7122580.0,7.256752e+06,7.389719e+06,7.505974e+06,7.609673e+06,7.728139e+06,7.848202e+06,7.938794e+06,8.025321e+06,8.074426e+06,8.098875e+06,9.762954e+05,13.707047
65-79 Sénior,7315305.0,7.496579e+06,7.675785e+06,7.840684e+06,8.054623e+06,8.271145e+06,8.481158e+06,8.681559e+06,8.873134e+06,9.070451e+06,9.254488e+06,1.939183e+06,26.508578


## Variación porcentual anual

Hasta este punto se ha analizado la evolución total de cada segmento entre el primer y el último año del periodo de estudio. Sin embargo, este enfoque no permite identificar si el crecimiento o la disminución de la población se produce de forma constante o si existen años en los que la variación se acelera o se ralentiza.

Por ello, se calcula la **variación porcentual anual**, que mide el cambio relativo de la población entre dos años consecutivos.

La variación porcentual anual se calcula mediante la siguiente expresión:

\[
Variación\ (\%)=\frac{Población_{año\ actual}-Población_{año\ anterior}}
{Población_{año\ anterior}}\times100
\]

Este análisis permite:

- Identificar años con mayores incrementos o descensos de población.
- Detectar posibles aceleraciones o desaceleraciones en el crecimiento de cada segmento.
- Analizar si la evolución demográfica sigue una tendencia estable o presenta cambios significativos a lo largo del periodo estudiado.

La información obtenida complementa el análisis del crecimiento total y facilita una interpretación más detallada de la evolución de cada segmento poblacional.

In [23]:
# Crear una copia
df_variacion = df_segmentos_edad.copy()

# Ordenar por segmento y año
df_variacion = df_variacion.sort_values(
    ["segmento_edad", "periodo"]
)

# Calcular la variación porcentual anual
df_variacion["variacion_anual_%"] = (
    df_variacion
    .groupby("segmento_edad")["poblacion"]
    .pct_change() * 100
)

# Redondear
df_variacion["variacion_anual_%"] = (
    df_variacion["variacion_anual_%"]
    .round(2)
)

display(df_variacion)

,periodo,segmento_edad,poblacion,total,porcentaje,variacion_anual_%
0,2026,18-24 Jóvenes,3.878454e+06,3.860349e+07,10.046901,NaN
6,2027,18-24 Jóvenes,4.002649e+06,3.911572e+07,10.232839,3.20
12,2028,18-24 Jóvenes,4.100102e+06,3.959168e+07,10.355969,2.43
18,2029,18-24 Jóvenes,4.166547e+06,4.001232e+07,10.413160,1.62
24,2030,18-24 Jóvenes,4.206732e+06,4.041818e+07,10.408019,0.96
...,...,...,...,...,...,...
41,2032,65-79 Sénior,8.481158e+06,4.111770e+07,20.626539,2.54
47,2033,65-79 Sénior,8.681559e+06,4.138814e+07,20.975958,2.36
53,2034,65-79 Sénior,8.873134e+06,4.162399e+07,21.317353,2.21
59,2035,65-79 Sénior,9.070451e+06,4.182807e+07,21.685081,2.22


## Tasa de Crecimiento Anual Compuesta (CAGR)

Aunque el crecimiento absoluto y porcentual muestran la variación total entre el primer y el último año del estudio, no indican cuál ha sido el ritmo medio de crecimiento de cada segmento a lo largo del periodo.

Para ello se calcula la **Tasa de Crecimiento Anual Compuesta (Compound Annual Growth Rate, CAGR)**.

El CAGR representa el porcentaje medio al que habría crecido (o disminuido) un segmento cada año si su evolución hubiera sido constante durante todo el periodo analizado.

Su cálculo se realiza mediante la siguiente expresión:

\[
CAGR=\left(\frac{Valor\ Final}{Valor\ Inicial}\right)^{1/n}-1
\]

donde:

- **Valor Inicial:** población del segmento en el primer año del estudio.
- **Valor Final:** población del segmento en el último año.
- **n:** número de años transcurridos.

Este indicador permite comparar el ritmo de crecimiento de segmentos con tamaños muy diferentes y facilita identificar cuáles presentan una evolución demográfica más favorable desde una perspectiva de negocio.

**La interpretación sería:**

Un CAGR positivo indica que el segmento crece de forma sostenida cada año.
Un CAGR negativo indica una disminución progresiva del segmento.
Cuanto mayor sea el valor del CAGR, mayor será el ritmo medio de crecimiento anual.

In [24]:
# Número de años entre la primera y la última proyección
n_anios = 2036 - 2026

# Cálculo del CAGR
df_crecimiento["CAGR (%)"] = (
    (
        df_crecimiento[2036] / df_crecimiento[2026]
    ) ** (1 / n_anios) - 1
) * 100

# Redondear a dos decimales
df_crecimiento["CAGR (%)"] = df_crecimiento["CAGR (%)"].round(2)

display(df_crecimiento)

periodo,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,crecimiento_absoluto,crecimiento_%,CAGR (%)
segmento_edad,,,,,,,,,,,,,,
18-24 Jóvenes,3878454.0,4.002649e+06,4.100102e+06,4.166547e+06,4.206732e+06,4.223637e+06,4.195619e+06,4.165050e+06,4.105946e+06,4.051151e+06,3.980563e+06,1.021087e+05,2.632716,0.26
25-34 Jóvenes adultos,5689845.0,5.804328e+06,5.913587e+06,6.044268e+06,6.184294e+06,6.321412e+06,6.458615e+06,6.577189e+06,6.706231e+06,6.792953e+06,6.850573e+06,1.160728e+06,20.399985,1.87
35-44 Adultos,6526108.0,6.479453e+06,6.466621e+06,6.473977e+06,6.482370e+06,6.500020e+06,6.525927e+06,6.562390e+06,6.590409e+06,6.632945e+06,6.689109e+06,1.630013e+05,2.497680,0.25
45-54 Adultos consolidados,8071195.0,8.075960e+06,8.045868e+06,7.980870e+06,7.880490e+06,7.755446e+06,7.608178e+06,7.463157e+06,7.322954e+06,7.206143e+06,7.101191e+06,-9.700041e+05,-12.018098,-1.27
55-64 Prejubilados,7122580.0,7.256752e+06,7.389719e+06,7.505974e+06,7.609673e+06,7.728139e+06,7.848202e+06,7.938794e+06,8.025321e+06,8.074426e+06,8.098875e+06,9.762954e+05,13.707047,1.29
65-79 Sénior,7315305.0,7.496579e+06,7.675785e+06,7.840684e+06,8.054623e+06,8.271145e+06,8.481158e+06,8.681559e+06,8.873134e+06,9.070451e+06,9.254488e+06,1.939183e+06,26.508578,2.38


## Ranking de segmentos poblacionales

Una vez analizada la evolución de cada segmento mediante el crecimiento absoluto, el crecimiento porcentual y la tasa de crecimiento anual compuesta (CAGR), resulta útil sintetizar los resultados en un ranking que facilite su interpretación desde una perspectiva de negocio.

El objetivo de este análisis es identificar qué segmentos presentan una evolución más favorable y cuáles muestran una pérdida de peso dentro de la población proyectada.

Para ello se ordenan los segmentos según tres indicadores:

- **Mayor crecimiento absoluto:** número de personas que gana o pierde cada segmento entre el primer y el último año del estudio.
- **Mayor crecimiento porcentual:** crecimiento relativo respecto al tamaño inicial del segmento.
- **Mayor peso poblacional:** porcentaje que representa cada segmento sobre la población total en el último año de la proyección.

Este ranking permite detectar qué grupos de población tendrán una mayor relevancia en el futuro y facilita la toma de decisiones estratégicas relacionadas con la segmentación de clientes, el diseño de nuevos productos y la asignación de recursos comerciales.

In [25]:
# ===========================================
# RANKING DE SEGMENTOS
# ===========================================

ranking = df_crecimiento.copy()

# Orden por crecimiento absoluto
ranking_abs = (
    ranking
    .sort_values("crecimiento_absoluto", ascending=False)
    [["crecimiento_absoluto","crecimiento_%","CAGR (%)"]]
)

print("🏆 Ranking por crecimiento absoluto")
display(ranking_abs)

# Orden por crecimiento porcentual
ranking_pct = (
    ranking
    .sort_values("crecimiento_%", ascending=False)
    [["crecimiento_absoluto","crecimiento_%","CAGR (%)"]]
)

print("🏆 Ranking por crecimiento porcentual")
display(ranking_pct)

🏆 Ranking por crecimiento absoluto


periodo,crecimiento_absoluto,crecimiento_%,CAGR (%)
segmento_edad,,,
65-79 Sénior,1.939183e+06,26.508578,2.38
25-34 Jóvenes adultos,1.160728e+06,20.399985,1.87
55-64 Prejubilados,9.762954e+05,13.707047,1.29
35-44 Adultos,1.630013e+05,2.497680,0.25
18-24 Jóvenes,1.021087e+05,2.632716,0.26
45-54 Adultos consolidados,-9.700041e+05,-12.018098,-1.27


🏆 Ranking por crecimiento porcentual


periodo,crecimiento_absoluto,crecimiento_%,CAGR (%)
segmento_edad,,,
65-79 Sénior,1.939183e+06,26.508578,2.38
25-34 Jóvenes adultos,1.160728e+06,20.399985,1.87
55-64 Prejubilados,9.762954e+05,13.707047,1.29
18-24 Jóvenes,1.021087e+05,2.632716,0.26
35-44 Adultos,1.630013e+05,2.497680,0.25
45-54 Adultos consolidados,-9.700041e+05,-12.018098,-1.27


## Análisis de tendencias mediante regresión lineal

Hasta este punto se ha analizado la evolución de la población mediante indicadores descriptivos como el crecimiento absoluto, el crecimiento porcentual y la tasa de crecimiento anual compuesta (CAGR). Sin embargo, estos indicadores no permiten determinar si la evolución observada responde a una tendencia consistente a lo largo del tiempo.

Para ello se aplica una **regresión lineal simple**, considerando:

- **Variable independiente (X):** año de la proyección.
- **Variable dependiente (Y):** población de cada segmento.

La regresión lineal ajusta una recta que representa la evolución media de la población en el tiempo y proporciona varios indicadores estadísticos:

- **Pendiente (Slope):** indica el cambio medio anual de la población. Una pendiente positiva implica crecimiento, mientras que una pendiente negativa indica una disminución del segmento.
- **Coeficiente de determinación (R²):** mide qué proporción de la variabilidad de la población es explicada por la tendencia temporal. Valores próximos a 1 indican un ajuste muy bueno.
- **Valor p (p-value):** permite evaluar si la tendencia observada es estadísticamente significativa. Se considera significativa cuando el valor p es inferior a 0,05.

Este análisis permite aportar evidencia estadística sobre la evolución de cada segmento poblacional y refuerza las conclusiones obtenidas mediante los indicadores descriptivos.

### Interpretación de los resultados

La pendiente indica el cambio medio anual de la población en cada segmento.

- Una **pendiente positiva** refleja un incremento sostenido del número de personas en ese segmento.
- Una **pendiente negativa** indica una disminución progresiva.

El coeficiente **R²** muestra la calidad del ajuste del modelo lineal. Valores próximos a 1 indican que la evolución del segmento sigue una tendencia temporal muy estable.

Por último, el **valor p** permite evaluar la significación estadística de la tendencia. Cuando **p < 0,05**, puede afirmarse que la evolución observada es estadísticamente significativa y difícilmente atribuible al azar.

Este análisis proporciona evidencia estadística que complementa los indicadores descriptivos, permitiendo fundamentar con mayor solidez las decisiones relacionadas con la redefinición de los segmentos de clientes.

from scipy.stats import linregress
import pandas as pd

# ==============================================
# REGRESIÓN LINEAL POR SEGMENTO DE EDAD
# ==============================================

resultados_regresion = []

for segmento in df_segmentos_edad["segmento_edad"].unique():

    datos = (
        df_segmentos_edad[
            df_segmentos_edad["segmento_edad"] == segmento
        ]
        .sort_values("periodo")
    )

    regresion = linregress(
        datos["periodo"],
        datos["poblacion"]
    )

    resultados_regresion.append({
        "Segmento": segmento,
        "Pendiente": regresion.slope,
        "Intercepto": regresion.intercept,
        "R²": regresion.rvalue**2,
        "p-value": regresion.pvalue
    })

df_regresion = pd.DataFrame(resultados_regresion)

# Redondear resultados
df_regresion["Pendiente"] = df_regresion["Pendiente"].round(0)
df_regresion["R²"] = df_regresion["R²"].round(4)
df_regresion["p-value"] = df_regresion["p-value"].round(6)

display(df_regresion)

# Clasificar la población en generaciones
## Segmentación por generaciones

Además de agrupar la población por intervalos de edad, resulta interesante analizar la evolución de las principales generaciones.

Esta clasificación agrupa a las personas según el periodo histórico en el que nacieron, ya que cada generación suele presentar patrones de consumo, uso de tecnología y comportamiento financiero diferentes.

En este análisis se consideran:

- Generación Z.
- Millennials.
- Generación X.
- Baby Boomers.

El objetivo es observar cómo evolucionará el peso de cada generación dentro de la población y anticipar posibles cambios en las necesidades del mercado.

Esta segmentación resulta especialmente útil para adaptar productos y estrategias comerciales dirigidas a perfiles generacionales concretos.

In [28]:
df_generaciones = df_poblacion_larga.copy()

condiciones = [
    df_generaciones["edad_num"].between(18,27),
    df_generaciones["edad_num"].between(28,43),
    df_generaciones["edad_num"].between(44,59),
    df_generaciones["edad_num"].between(60,79)
]

etiquetas = [
    "Generación Z",
    "Millennials",
    "Generación X",
    "Baby Boomers"
]

df_generaciones["generacion"] = np.select(
    condiciones,
    etiquetas,
    default="Otros"
)

df_generaciones = (
    df_generaciones
    .groupby(["periodo","generacion"], observed=True)["poblacion"]
    .sum()
    .reset_index()
)

display(df_generaciones)

,periodo,generacion,poblacion
0,2026,Baby Boomers,1.071484e+07
1,2026,Generación X,1.253690e+07
2,2026,Generación Z,5.546144e+06
3,2026,Millennials,9.805604e+06
4,2027,Baby Boomers,1.097347e+07
5,2027,Generación X,1.258040e+07
6,2027,Generación Z,5.726592e+06
7,2027,Millennials,9.835257e+06
8,2028,Baby Boomers,1.123279e+07
9,2028,Generación X,1.257812e+07


## Índice de envejecimiento

El índice de envejecimiento mide la relación entre la población de mayor edad y la población joven.

Se calcula comparando el número de personas de 65 años o más con el número de personas menores de 25 años.

Este indicador permite evaluar el grado de envejecimiento de la población y analizar cómo evolucionará la estructura demográfica del país.

Un aumento del índice implica una mayor presencia de población sénior, lo que puede modificar la demanda de determinados productos y servicios financieros, especialmente aquellos relacionados con el ahorro, la jubilación o la planificación patrimonial.

In [ ]:
df_indice = df_poblacion_larga.copy()

mayores = (
    df_indice[df_indice["edad_num"]>=65]
    .groupby("periodo")["poblacion"]
    .sum()
)

jovenes = (
    df_indice[df_indice["edad_num"]<=24]
    .groupby("periodo")["poblacion"]
    .sum()
)

indice_envejecimiento = (
    mayores
    / jovenes
    *100
)

display(indice_envejecimiento)

## Índice de dependencia demográfica

El índice de dependencia relaciona la población considerada dependiente con la población en edad potencialmente activa.

En este análisis se considera población dependiente a:

- Personas menores de 25 años.
- Personas de 65 años o más.

La población activa corresponde al grupo comprendido entre los 25 y los 64 años.

Este indicador permite estimar cómo evolucionará la presión demográfica sobre la población en edad laboral y ayuda a comprender cambios futuros en el consumo, el ahorro y las necesidades financieras de los hogares.

In [ ]:
dependientes = (
    df_indice[
        (df_indice["edad_num"]<=24) |
        (df_indice["edad_num"]>=65)
    ]
    .groupby("periodo")["poblacion"]
    .sum()
)

activos = (
    df_indice[
        df_indice["edad_num"].between(25,64)
    ]
    .groupby("periodo")["poblacion"]
    .sum()
)

indice_dependencia = (
    dependientes
    / activos
    *100
)

display(indice_dependencia)

## Índices demográficos complementarios

Además del índice de envejecimiento y del índice de dependencia, se calculan tres indicadores que permiten describir la estructura de la población desde diferentes perspectivas.

Estos índices expresan el peso relativo de determinados grupos de edad sobre el total de la población proyectada, facilitando la interpretación de la evolución demográfica y apoyando la toma de decisiones estratégicas.

Los indicadores calculados son:

- **Índice de juventud:** porcentaje de población entre 18 y 34 años respecto al total de la población. Este indicador permite estimar el peso de los clientes potenciales más jóvenes.

- **Índice de población activa:** porcentaje de población entre 25 y 64 años respecto al total de la población. Representa la población en edad laboral y constituye el principal mercado potencial para productos financieros como hipotecas, inversión o seguros.

- **Índice sénior:** porcentaje de población de 65 años o más respecto al total de la población. Este indicador permite cuantificar el proceso de envejecimiento y anticipar una mayor demanda de productos relacionados con el ahorro, la jubilación y la planificación patrimonial.

Estos indicadores complementan el análisis realizado anteriormente y permiten evaluar cómo cambia la composición de la población a lo largo del horizonte temporal de las proyecciones.

In [ ]:
# =====================================================
# ÍNDICES DEMOGRÁFICOS COMPLEMENTARIOS
# =====================================================

# Población total por año
total_poblacion = (
    df_poblacion_larga
    .groupby("periodo")["poblacion"]
    .sum()
)

# Índice de juventud (18-34 años)
indice_juventud = (
    df_poblacion_larga[
        df_poblacion_larga["edad_num"].between(18, 34)
    ]
    .groupby("periodo")["poblacion"]
    .sum()
    .div(total_poblacion)
    .mul(100)
)

# Índice de población activa (25-64 años)
indice_activa = (
    df_poblacion_larga[
        df_poblacion_larga["edad_num"].between(25, 64)
    ]
    .groupby("periodo")["poblacion"]
    .sum()
    .div(total_poblacion)
    .mul(100)
)

# Índice sénior (65 años o más)
indice_senior = (
    df_poblacion_larga[
        df_poblacion_larga["edad_num"] >= 65
    ]
    .groupby("periodo")["poblacion"]
    .sum()
    .div(total_poblacion)
    .mul(100)
)

# Crear tabla resumen
df_indices_demograficos = pd.DataFrame({
    "Índice de juventud (%)": indice_juventud,
    "Índice de población activa (%)": indice_activa,
    "Índice sénior (%)": indice_senior
})

# Redondear resultados
df_indices_demograficos = df_indices_demograficos.round(2)

display(df_indices_demograficos)

# Estadistica descriptiva

In [ ]:
df_poblacion_larga.groupby("periodo")["poblacion"].describe()